# Champion vs Phase 3 vs Champion V2 — one-hour comparison

Use a **GPU Colab runtime**. This notebook gives each method the same topology and optimizer seed, 50 frequencies, microbatch size 1, and **60 minutes of scored Objective time**. Compilation and setup can add wall time (up to 15 minutes per method); the three runs execute sequentially. It saves each run and a comparison report to Drive, and resumes completed runs with matching settings.

The JAX GPU pool is capped at 16,000 MiB and a monitor stops a run if observed GPU use reaches 19,000 MiB (below 20 GB decimal). On a GPU larger than 20 GB, this monitor is best effort rather than a hardware guarantee. Choose a **T4 16 GB** runtime if the 20 GB limit must be physically enforced.

The bundled “N7 local” archive has **not** been verified as the exact Phase 3 portal upload. Set `PHASE3_ZIP` to that downloaded archive if you have it; otherwise the comparison labels N7 as an unverified local proxy. A single one-hour topology is a screening test, not proof of final four-hour performance.

In [ ]:
# Set the kit path in Drive, or leave blank to upload the ZIP when prompted.
KIT_PATH = ""
OUTPUT_DIR = "/content/drive/MyDrive/L2D_2026_20GB_1H_compare"

# Optional: exact downloaded Phase 3 submission ZIP in Drive.
PHASE3_ZIP = ""
PHASE3_SHA256 = ""  # Fill this if you have a trusted hash for that ZIP.
PHASE3_PORTAL_VERIFIED = False  # True only when you verified its portal provenance.

SEED_SOURCE = "holdout"  # "holdout" or "round3"
SEED_INDEX = 0
KIT_SHA256 = "e5844e98faf64c0c2748dcd3ccd819ab78b86285dea534fbd6f87ff6419793c2"

## Load the experiment kit

Upload [`l2d_20gb_1h_compare_kit.zip`](../dist/l2d_20gb_1h_compare_kit.zip) from this project, or put it in Drive and set `KIT_PATH`. The kit contains the three archives, seeds, benchmark, and memory guard. It does not upload anything to the competition portal.

In [ ]:
import hashlib, json, pathlib, subprocess, sys, zipfile
from google.colab import drive, files
from IPython.display import display, Markdown

drive.mount("/content/drive")
out = pathlib.Path(OUTPUT_DIR)
out.mkdir(parents=True, exist_ok=True)
if not KIT_PATH:
    uploaded = files.upload()
    assert len(uploaded) == 1, "Select only the experiment kit ZIP"
    KIT_PATH = "/content/" + next(iter(uploaded))
kit = pathlib.Path(KIT_PATH)
assert kit.is_file(), kit

def sha256(path):
    digest = hashlib.sha256()
    with pathlib.Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

assert sha256(kit) == KIT_SHA256, "Kit checksum mismatch: upload the matching kit"
root = pathlib.Path("/content/l2d_20gb_1h_kit")
root.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(kit) as archive:
    for entry in archive.infolist():
        path = pathlib.PurePosixPath(entry.filename)
        assert not path.is_absolute() and ".." not in path.parts, "Unsafe kit member"
    archive.extractall(root)
manifest = json.loads((root / "manifest.json").read_text())
for name, expected_hash in manifest["files_sha256"].items():
    assert sha256(root / name) == expected_hash, "Kit member checksum mismatch: " + name

baseline = root / "artifacts/baselines/local_champion.zip"
candidate = root / "dist/champion_v2.zip"
phase3 = pathlib.Path(PHASE3_ZIP) if PHASE3_ZIP else root / "dist/n7_fsq.zip"
assert phase3.is_file(), phase3
if PHASE3_SHA256:
    assert sha256(phase3) == PHASE3_SHA256.lower(), "Phase 3 ZIP checksum mismatch"
phase3_label = ("Phase 3 portal ZIP (user verified)" if PHASE3_ZIP and PHASE3_PORTAL_VERIFIED
                else "Phase 3 external ZIP (identity unverified)" if PHASE3_ZIP
                else "N7 local (Phase 3 identity unverified)")
print("Kit SHA-256:", sha256(kit))
for label, path in (("Champion", baseline), (phase3_label, phase3), ("Champion V2", candidate)):
    print(label, sha256(path))

## Install and verify the benchmark environment

The benchmark uses `dfbench==0.3.3` and its pinned GPU-enabled JAX 0.9.0.1 CUDA 12 build. This cell verifies a float64 GPU operation before any hour-long run. JAX is imported only in a fresh subprocess so each method gets the same memory settings.

In [ ]:
# Install the benchmark and its pinned GPU-enabled JAX build.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "jax", "jaxlib",
                "jax-cuda12-plugin", "jax-cuda12-pjrt",
                "jax-cuda13-plugin", "jax-cuda13-pjrt"], check=False,
               stdout=subprocess.DEVNULL)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "dfbench[cuda12]==0.3.3", "optax", "osqp>=1,<2", "scipy>=1.15",
                "h5py", "matplotlib"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade",
                "jax[cuda12]==0.9.0.1"], check=True)
print(subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total,memory.used",
                               "--format=csv,noheader,nounits"], text=True).strip())
sys.path.insert(0, str(root / "tools"))
import colab_20gb_compare as gpu_check
card = gpu_check.gpu_card()
pool = gpu_check.pool_size(card, 19000, 16000)
print("GPU float64 probe:", gpu_check.probe_gpu(gpu_check.run_environment(card, pool)))

## Check the plan, then run

The plan verifies the ZIPs and seed before using GPU time. Each run writes progress and results directly to Drive. Rerun this cell after a disconnect to reuse any completed, matching methods.

In [ ]:
assert SEED_SOURCE in {"holdout", "round3"}
seed_file = (root / "analysis/final_holdout_seeds.csv" if SEED_SOURCE == "holdout"
             else root / "competition_data/round3/seeds.csv")
command = [sys.executable, str(root / "tools/colab_20gb_compare.py"),
           "--baseline", str(baseline), "--phase3", str(phase3),
           "--phase3-label", phase3_label, "--candidate", str(candidate),
           "--seed-file", str(seed_file), "--seed-index", str(SEED_INDEX),
           "--minutes", "60", "--cap-mib", "19000", "--pool-limit-mib", "16000",
           "--out", str(out)]
if PHASE3_ZIP and PHASE3_PORTAL_VERIFIED:
    command.append("--phase3-verified")
subprocess.run(command + ["--dry-run"], check=True)

In [ ]:
# Stream output to the notebook and save the full log in Drive.
import signal
runner_log = out / "runner.log"
with runner_log.open("w") as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1,
                               start_new_session=True)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        runner_exit_code = process.wait()
    except KeyboardInterrupt:
        process.send_signal(signal.SIGINT)
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
print("Runner exit code:", runner_exit_code)
print("Runner log:", runner_log)
print("Results saved in:", out)
if runner_exit_code:
    print("The run stopped early. Check runner.log and any runs/*/driver.log.")

## Compare the results

Lower best feasible loss is better. The report gives elapsed Objective time, evaluation counts, observed peak GPU memory, and the provenance label for each archive.

In [ ]:
report_file = out / "comparison.json"
if not report_file.is_file():
    log_file = out / "runner.log"
    if log_file.is_file():
        print("Runner log (last 80 lines):")
        print("\n".join(log_file.read_text(errors="replace").splitlines()[-80:]))
    raise RuntimeError("No comparison report was produced. Run the preceding cell and inspect runner.log above.")
report = json.loads(report_file.read_text())
display(Markdown((out / "comparison.md").read_text()))
print("Detailed files:", out / "comparison.csv", out / "comparison.json")

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 5))
for row in report["results"]:
    progress = pathlib.Path(row["run_dir"]) / "progress.jsonl"
    if not progress.is_file():
        continue
    points = []
    for line in progress.read_text().splitlines():
        try:
            point = json.loads(line)
            if point.get("elapsed") is not None and point.get("best_feasible") is not None:
                points.append((point["elapsed"] / 60, point["best_feasible"]))
        except (ValueError, TypeError):
            continue
    if points:
        ax.step([point[0] for point in points], [point[1] for point in points],
                where="post", label=row["method"])
ax.set(xlabel="Scored Objective minutes", ylabel="Best feasible loss (lower is better)",
       title="One-hour paired comparison", xlim=(0, 60))
ax.grid(alpha=0.25)
if ax.lines:
    ax.legend()
fig.tight_layout()
chart = out / "comparison.png"
fig.savefig(chart, dpi=150)
plt.show()
print("Chart saved:", chart)